📅 **论文年份 (Year):2020 年**  
*Dense Passage Retrieval for Open-Domain Question Answering — Karpukhin et al.*

# Paper 28: Dense Passage Retrieval for Open-Domain Question Answering(论文 28:面向开放域问答的稠密段落检索)
## Vladimir Karpukhin, Barlas Oğuz, Sewon Min, et al., Meta AI (2020)(Vladimir Karpukhin、Barlas Oğuz、Sewon Min 等,Meta AI,2020)

### Dense Passage Retrieval (DPR)(稠密段落检索)

Learn dense embeddings for questions and passages. Retrieve via similarity in embedding space. Beats BM25!

为问题(question)和段落(passage)学习稠密嵌入(dense embeddings)。通过在嵌入空间中计算相似度来进行检索。效果超越 BM25!

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 想让机器回答"珠穆朗玛峰有多高"这类问题,第一步是从海量文档里找出可能包含答案的段落。传统方法(如 BM25)靠"对关键词"来找:问题和段落里出现相同的词才算匹配。可一旦你换了说法——比如问"最高的山峰",而文章里写的是"海拔最高的山"——关键词对不上,就找不到了。这篇论文要解决的正是这种"意思一样、用词不同就检索失败"的问题。

**💡 主要贡献:** 论文提出了 DPR(稠密段落检索):不再比对字面上的词,而是把问题和段落都变成一串数字(向量),意思相近的文本在这个"数字空间"里彼此靠近,就像把书按内容而不是按书名字母排到书架上。作者还证明,不需要海量数据,只用几万个问答对就能训练出效果大幅超越 BM25 的检索器(Top-20 准确率提升约 9-19 个百分点)。

**🔧 方法:** DPR 用两个 BERT 编码器(即"双编码器"):一个负责把问题变成向量,另一个负责把段落变成向量,向量做点积就得到相似度分数。训练时采用对比学习:让问题和正确段落的分数越高越好,和错误段落的分数越低越好。一个巧妙的省力技巧是"批内负样本"——同一批训练数据里,别人的正确答案就直接当作你的错误答案,不用额外找反例。检索时先离线算好所有段落的向量并建索引(如 FAISS),用户提问时只需算一次问题向量,再快速查找最相近的段落。

**🌟 意义:** DPR 证明了"学出来的语义检索"可以打败几十年的关键词检索传统,直接推动了检索技术的范式转变。今天大家熟悉的 RAG(检索增强生成)——让 ChatGPT 这类大模型先查资料再回答——其检索环节正是建立在 DPR 开创的稠密检索思路之上。可以说,向量数据库、语义搜索等如今火热的技术,都能追溯到这篇论文打下的地基。

## 🎯 核心结论 (Key Takeaways)

- **稠密检索大幅超越关键词检索:** DPR 论文的核心发现是,把问题和段落都编码成向量再比相似度,效果远超传统 BM25——在 Natural Questions 上 Top-20 准确率 78.4% vs 59.1%,在 WebQuestions(75.0% vs 55.0%)和 TREC(79.4% vs 70.9%)上同样领先约 9-19 个百分点。
- **不需要海量数据:** 只用约 5.9 万个问答对微调两个 BERT 编码器,就足以训练出打败 BM25 的检索器,推翻了"语义检索必须靠超大规模预训练"的假设。
- **省力的训练技巧——批内负样本 (in-batch negatives):** 同一批次(batch=128)里,别人问题的正确段落直接当作你的负样本,不用额外挖反例就能获得大量对比信号;本 notebook 用 3 个问题的小批次演示了这一 InfoNCE 对比损失的计算过程。
- **本 notebook 的完整流水线:** 用简化 RNN 双编码器 + 点积最大内积搜索 (MIPS) 走通了"编码语料 → 检索 top-k → 与 BM25 逐题对比 → Recall@k / MRR 定量评估"全流程;由于演示用的编码器未经训练(权重随机),稠密检索结果基本是瞎猜,而 BM25 无需训练就能靠关键词命中——这恰好说明稠密检索的威力完全来自"训练",训练好之后才能反超。
- **各有短板,实践中常混合使用:** 稠密检索能匹配"tallest mountain"和"highest peak"这类同义改写,但对稀有实体的精确匹配不如 BM25,且需要为全部段落存向量、语料更新要重新编码,所以工业界常用 BM25 + 稠密检索的混合方案。
- **一句话带走:** 检索从"对字面"走向"对意思"始于 DPR——今天的 RAG、向量数据库和语义搜索,底层用的正是这套"双编码器 + 向量近邻查找"的思路。


## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为:关键词匹配统治了搜索几十年(从图书馆检索到 Google 早期),想打败 BM25 得靠更精巧的关键词技巧。** 但这篇论文发现:把整句话"压扁"成一个稠密向量再比内积,反而大幅碾压 BM25——Natural Questions 上 Top-20 准确率 78.4% vs 59.1%,领先近 20 个百分点。问"最高的山峰"、文中写"海拔最高的山",一个词都不重叠也能搜到,因为比的是"意思"而不是"字面"。

- **常识认为:对比学习需要费力去挖掘大量高质量负样本,负例越多、挖得越精,训练才越好。** 但 DPR 发现"白捡"的负例就够了:同一个 batch 里,别人的正确段落天然就是你的错误段落(in-batch negatives)——一个 batch 内的编码复用一下,不花任何额外计算就得到 B-1 个负样本。本笔记本的 `contrastive_loss` 实验正是演示这一点:只靠这种"顺手牵羊"的负例,损失就能把正确段落的得分推上去。

- **常识认为:想让神经检索打败调了几十年的 BM25,得喂海量标注数据。** 但论文发现只要约 5.9 万个问答对微调 BERT 就够了,根本不需要百万级标注。反过来,训练也不可省略:本笔记本用未训练(随机初始化)的编码器做稠密检索,Recall@1 基本是随机水平,甚至不如简单的 BM25——稠密检索的"语义理解力"不是架构自带的,而是完全靠对比训练学出来的。

- **常识认为:在几百万个段落里逐一算相似度肯定慢得没法用,语义搜索只能是实验室玩具。** 但因为相似度就是一个简单的内积,检索可以写成一次矩阵乘法(本笔记本的 `retrieve_top_k` 只有一行 `np.dot`),再配合 FAISS 这类最大内积搜索(MIPS)索引,千万级语料也能毫秒级返回——"暴力"的向量检索反而成了今天 RAG 系统的标配。

#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库,并固定随机种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(数值计算)、`matplotlib.pyplot`(画图)、`Counter`(词频统计,后面 BM25 会用到)和 `re`(正则表达式)。
- 调用 `np.random.seed(42)` 固定随机数种子——就像掷骰子前先"锁定"骰子,让每次运行生成的随机权重都一样,结果可复现。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

np.random.seed(42)

## Dual Encoder Architecture(双编码器架构)

```
Question → Encoder_Q → q (dense vector)
Passage  → Encoder_P → p (dense vector)

Similarity: sim(q, p) = q · p  (dot product)
```

问题(Question)和段落(Passage)分别由各自的编码器映射为稠密向量,相似度通过点积(dot product)计算:sim(q, p) = q · p。

#### 💻 代码解读

**做什么:** 定义一个简化版的文本编码器 `SimpleTextEncoder`,把一句话(一串词的编号)"压缩"成一个固定长度的稠密向量——这是 DPR 双编码器(dual encoder)的核心零件。真实论文用的是 BERT,这里用简单 RNN 代替来演示原理。

**怎么做:**
- `__init__` 里随机初始化三组参数:词嵌入表 `embeddings`(每个词对应一个向量)、RNN 权重 `W_xh`/`W_hh`/`b_h`(负责按顺序"读"句子)、输出投影 `W_out`。
- `encode` 方法逐个读入 token:查出词向量后用 `tanh` 更新隐藏状态 `h`,就像人一个词一个词地读句子、不断更新脑中的理解。
- 读完整句后用 `W_out` 投影得到最终向量,并做 L2 归一化(把向量长度归一),这样两个向量的点积就等于余弦相似度。
- 最后创建两个独立的编码器:`question_encoder`(编码问题)和 `passage_encoder`(编码段落)——"双编码器"就是问题和段落各用各的编码器;再用一段测试 token 验证输出形状和相似度。

In [ ]:
# DPR的核心组件:把文本编码成稠密向量。论文中用BERT,这里用简单RNN示意
class SimpleTextEncoder:
    """Simplified text encoder (in practice: use BERT)"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        # Embeddings
        # 词嵌入查找表,形状(vocab_size, embedding_dim);乘0.01做小随机初始化避免激活饱和
        self.embeddings = np.random.randn(vocab_size, embedding_dim) * 0.01
        
        # Simple RNN weights
        self.W_xh = np.random.randn(hidden_dim, embedding_dim) * 0.01
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_h = np.zeros((hidden_dim, 1))
        
        # Output projection
        self.W_out = np.random.randn(hidden_dim, hidden_dim) * 0.01
    
    def encode(self, token_ids):
        """
        Encode sequence of token IDs to dense vector
        Returns: dense embedding (hidden_dim,)
        """
        # 隐状态初始化为零向量,形状(hidden_dim, 1),将随序列逐词更新
        h = np.zeros((self.hidden_dim, 1))
        
        # Process tokens
        for token_id in token_ids:
            # Lookup embedding
            # 按行号取出该词的嵌入,reshape(-1,1)变成列向量(embedding_dim, 1)便于矩阵乘
            x = self.embeddings[token_id].reshape(-1, 1)
            
            # RNN step
            # 经典RNN更新:h_t = tanh(W_xh·x_t + W_hh·h_{t-1} + b),把当前词信息融入历史状态
            h = np.tanh(np.dot(self.W_xh, x) + np.dot(self.W_hh, h) + self.b_h)
        
        # Final representation (CLS-like)
        # 用最后一个隐状态代表整句(类似BERT的[CLS]向量),flatten把(hidden_dim,1)压成一维
        output = np.dot(self.W_out, h).flatten()
        
        # L2 normalize for cosine similarity
        # L2归一化后点积等价于余弦相似度;+1e-8防止零向量导致除零
        output = output / (np.linalg.norm(output) + 1e-8)
        
        return output

# Create encoders
vocab_size = 1000
embedding_dim = 64
hidden_dim = 128

# DPR是"双塔"结构:问题和文档各用一个独立编码器,两者只在最后通过点积交互
question_encoder = SimpleTextEncoder(vocab_size, embedding_dim, hidden_dim)
passage_encoder = SimpleTextEncoder(vocab_size, embedding_dim, hidden_dim)

# Test
test_tokens = [10, 25, 37, 42]
q_emb = question_encoder.encode(test_tokens)
p_emb = passage_encoder.encode(test_tokens)

print(f"Question embedding shape: {q_emb.shape}")
print(f"Passage embedding shape: {p_emb.shape}")
print(f"Similarity (dot product): {np.dot(q_emb, p_emb):.4f}")

## Synthetic QA Dataset(合成问答数据集)

#### 💻 代码解读

**做什么:** 定义一个简单的分词器 `SimpleTokenizer`,并手工构造一个小型问答数据集(8 个段落 + 5 个问题),作为后面检索实验的"迷你图书馆"。

**怎么做:**
- `SimpleTokenizer.tokenize` 把文本转小写、按空格切词,遇到没见过的词就分配一个新编号(存进 `word_to_id` 字典),返回整句的编号列表——相当于给每个单词发一张"身份证"。
- `passages` 列表存放 8 个知识段落(埃菲尔铁塔、长城、珠峰等)。
- `questions` 列表存放 5 个问题,每个问题都附带"标准答案":正确段落的下标(如"最高的山是什么?"对应第 5 段珠峰)。
- 用分词器把所有段落和问题转成 token 编号,并打印示例和词表大小。

In [ ]:
class SimpleTokenizer:
    """Simple word tokenizer"""
    def __init__(self):
        self.word_to_id = {}
        self.id_to_word = {}
        self.next_id = 0
    
    def tokenize(self, text):
        """Convert text to token IDs"""
        words = text.lower().split()
        token_ids = []
        
        # 边遍历边建词表:遇到新词就分配一个自增ID(真实系统会用固定的BERT词表)
        for word in words:
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1
            token_ids.append(self.word_to_id[word])
        
        return token_ids

# Create synthetic dataset
passages = [
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
    "The Great Wall of China is a series of fortifications in northern China.",
    "The Statue of Liberty is a colossal neoclassical sculpture in New York.",
    "The Colosseum is an oval amphitheatre in the centre of Rome, Italy.",
    "The Taj Mahal is an ivory-white marble mausoleum in Agra, India.",
    "Mount Everest is Earth's highest mountain above sea level.",
    "The Amazon River is the largest river by discharge volume of water.",
    "The Sahara is a desert on the African continent.",
]

# 每条数据是(问题, 正确文档下标),即检索任务的监督标签
questions = [
    ("What is the Eiffel Tower?", 0),  # (question, relevant_passage_idx)
    ("Where is the Great Wall located?", 1),
    ("What is the tallest mountain?", 5),
    ("Where is the Statue of Liberty?", 2),
    ("What is the largest river?", 6),
]

# Tokenize
tokenizer = SimpleTokenizer()

# 列表推导式批量分词;第二行用解包(q, idx)保留问题对应的正确文档下标
passage_tokens = [tokenizer.tokenize(p) for p in passages]
question_tokens = [(tokenizer.tokenize(q), idx) for q, idx in questions]

print("Sample passage:")
print(f"Text: {passages[0]}")
print(f"Tokens: {passage_tokens[0][:10]}...")
print(f"\nVocabulary size: {tokenizer.next_id}")

## Encode Corpus and Questions(编码语料库与问题)

#### 💻 代码解读

**做什么:** 用真实的词表大小重新创建问题编码器和段落编码器,然后把所有段落和问题都编码成稠密向量,建立一个小型"向量库"。

**怎么做:**
- 从 `tokenizer.next_id` 读出实际词表大小,重新初始化 `question_encoder` 和 `passage_encoder`(嵌入维度 32、隐藏维度 64)。
- 循环调用 `passage_encoder.encode` 把 8 个段落逐一编码,堆成矩阵 `passage_embeddings`(8×64)——真实系统中这一步是离线预先算好的,相当于提前给图书馆每本书做好"索引卡"。
- 同样用 `question_encoder` 把 5 个问题编码成 `question_embeddings`(5×64)。
- 打印两个矩阵的形状确认无误。

In [ ]:
# Re-initialize encoders with correct vocab size
vocab_size = tokenizer.next_id
question_encoder = SimpleTextEncoder(vocab_size, embedding_dim=32, hidden_dim=64)
passage_encoder = SimpleTextEncoder(vocab_size, embedding_dim=32, hidden_dim=64)

# Encode all passages
# 关键思想:所有文档向量可以离线预先算好并建索引,查询时只需编码问题
passage_embeddings = []
for tokens in passage_tokens:
    emb = passage_encoder.encode(tokens)
    passage_embeddings.append(emb)
# 堆叠成矩阵,形状(n_passages, hidden_dim),方便后面一次矩阵乘算出所有相似度
passage_embeddings = np.array(passage_embeddings)

# Encode questions
question_embeddings = []
for tokens, _ in question_tokens:
    emb = question_encoder.encode(tokens)
    question_embeddings.append(emb)
question_embeddings = np.array(question_embeddings)

print(f"Passage embeddings: {passage_embeddings.shape}")
print(f"Question embeddings: {question_embeddings.shape}")

## Dense Retrieval via Maximum Inner Product Search (MIPS)(基于最大内积搜索(MIPS)的稠密检索)

#### 💻 代码解读

**做什么:** 实现稠密检索的核心操作——最大内积搜索(MIPS):给定一个问题向量,从所有段落向量里找出最相似的前 k 个。

**怎么做:**
- 定义 `retrieve_top_k` 函数:用一次矩阵点积 `np.dot(passage_embeddings, query_embedding)` 算出问题与每个段落的相似度分数,再用 `np.argsort` 从高到低排序,取前 k 个段落的下标和分数。
- 对 5 个问题逐一测试:检索 top-3 段落,并用 ✓/✗ 标记是否命中"标准答案"段落。
- 最后特别提示:编码器还没训练过(权重是随机的),所以此时检索结果基本是瞎猜——这正好为后面的"训练"环节做铺垫。

In [ ]:
def retrieve_top_k(query_embedding, passage_embeddings, k=3):
    """
    Retrieve top-k passages for query
    Uses dot product similarity (MIPS)
    """
    # Compute similarities
    # 矩阵乘一次算出查询与所有文档的点积:(n_passages, dim)·(dim,) -> (n_passages,)
    similarities = np.dot(passage_embeddings, query_embedding)
    
    # Get top-k indices
    # argsort默认升序,[::-1]反转为降序,再切片取前k个即相似度最高的k篇
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_scores = similarities[top_k_indices]
    
    return top_k_indices, top_k_scores

# Test retrieval
print("\nDense Retrieval Results:\n" + "="*80)
for i, (q_tokens, correct_idx) in enumerate(question_tokens):
    question_text = questions[i][0]
    q_emb = question_embeddings[i]
    
    # Retrieve
    top_indices, top_scores = retrieve_top_k(q_emb, passage_embeddings, k=3)
    
    print(f"\nQ: {question_text}")
    print(f"Correct passage: #{correct_idx}")
    print(f"\nRetrieved (top-3):")
    # zip把下标和分数配对,enumerate(...,1)让名次从1开始计数
    for rank, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        is_correct = "✓" if idx == correct_idx else "✗"
        print(f"  {rank}. [{is_correct}] (score={score:.3f}) {passages[idx][:60]}...")

print("\n" + "="*80)
print("(Encoders are untrained, so results are random)")

## Training with In-Batch Negatives(使用批内负样本(in-batch negatives)训练)

#### 💻 代码解读

**做什么:** 演示 DPR 的训练目标——对比损失(InfoNCE)和"批内负样本"(in-batch negatives)技巧:让问题向量靠近正确段落、远离错误段落。

**怎么做:**
- 定义 `softmax` 函数(先减去最大值保证数值稳定)。
- 定义 `contrastive_loss`:把"问题·正确段落"的点积和"问题·错误段落"的点积放在一起做 softmax,再取正样本概率的负对数——正确段落得分越高、错误段落得分越低,损失就越小,好比做多选题时把正确选项的概率推向 1。
- 模拟一个大小为 3 的训练批次:对第 i 个问题,它配对的段落是正样本 `pos_emb`,批次里其他问题的段落直接拿来当负样本 `neg_embs`——这就是"批内负样本":不用额外去找反例,批次内互相"借用",非常省算力。
- 打印每个问题的损失和批次平均损失。

In [ ]:
def softmax(x):
    # 先减去最大值再取exp:softmax结果不变,但避免exp(大数)溢出为inf
    exp_x = np.exp(x - np.max(x))  # Numerical stability
    return exp_x / np.sum(exp_x)

def contrastive_loss(query_emb, positive_emb, negative_embs):
    """
    Contrastive loss (InfoNCE)
    
    L = -log( exp(q·p+) / (exp(q·p+) + Σ exp(q·p-)) )
    """
    # Positive score
    # 目标:让问题与正例文档的点积尽量大,与负例的点积尽量小
    pos_score = np.dot(query_emb, positive_emb)
    
    # Negative scores
    neg_scores = [np.dot(query_emb, neg_emb) for neg_emb in negative_embs]
    
    # All scores
    # 把正例放在第0位,后接所有负例,拼成一个"分类"的logits向量
    all_scores = np.array([pos_score] + neg_scores)
    
    # Softmax
    probs = softmax(all_scores)
    
    # Negative log likelihood (positive should be first)
    # InfoNCE等价于"在1个正例+N个负例中选中正例"的交叉熵;+1e-8防log(0)
    loss = -np.log(probs[0] + 1e-8)
    
    return loss

# Simulate training batch
batch_size = 3
batch_questions = question_embeddings[:batch_size]
batch_passages = passage_embeddings[:batch_size]

# In-batch negatives: for each question, other passages in batch are negatives
total_loss = 0
print("\nIn-Batch Negative Training:\n" + "="*80)
for i in range(batch_size):
    q_emb = batch_questions[i]
    pos_emb = batch_passages[i]  # Correct passage
    
    # Negatives: all other passages in batch
    # DPR的in-batch negatives技巧:同批次里其他问题的正例文档直接当作本问题的负例,免费获得负样本
    neg_embs = [batch_passages[j] for j in range(batch_size) if j != i]
    
    loss = contrastive_loss(q_emb, pos_emb, neg_embs)
    total_loss += loss
    
    print(f"Question {i}: loss = {loss:.4f}")

avg_loss = total_loss / batch_size
print(f"\nAverage batch loss: {avg_loss:.4f}")
print("\nIn-batch negatives: efficient hard negative mining!")

## Visualize Embedding Space(可视化嵌入空间)

#### 💻 代码解读

**做什么:** 把 64 维的嵌入向量投影到 2 维平面上画出来,直观展示问题和段落在"语义空间"里的分布。

**怎么做:**
- 定义 `project_2d` 函数:先对嵌入做均值中心化,再用 SVD(奇异值分解)取前两个主成分——相当于简化版 PCA,把高维空间"拍扁"成一张二维地图。
- 用 `np.vstack` 把段落向量和问题向量拼在一起统一投影,再拆回 `passage_2d` 和 `question_2d`。
- 用 matplotlib 画散点图:蓝色方块是段落(标注 P0-P7),红色圆点是问题(标注 Q0-Q4)。
- 用绿色虚线把每个问题和它的正确段落连起来——理想情况(训练好之后)问题应紧挨着自己的答案段落;现在模型未训练,点的位置基本随机。

In [ ]:
# Simple 2D projection (PCA-like)
def project_2d(embeddings):
    """Project high-dim embeddings to 2D (simplified PCA)"""
    # Mean center
    # axis=0沿样本维求均值;减均值是PCA的前提(广播:矩阵每行减同一向量)
    mean = np.mean(embeddings, axis=0)
    centered = embeddings - mean
    
    # Take first 2 principal components (simplified)
    # SVD分解X=U·S·Vt;U前2列乘对应奇异值即数据在前2个主成分上的坐标(等价于PCA投影)
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    projected = U[:, :2] * S[:2]
    
    return projected

# Project to 2D
# vstack把文档和问题向量竖着拼在一起做同一次投影,保证两者落在同一坐标系里可比
all_embeddings = np.vstack([passage_embeddings, question_embeddings])
projected = project_2d(all_embeddings)

# 按拼接顺序切片,把投影结果重新拆回文档部分和问题部分
passage_2d = projected[:len(passage_embeddings)]
question_2d = projected[len(passage_embeddings):]

# Visualize
plt.figure(figsize=(12, 10))

# Plot passages
plt.scatter(passage_2d[:, 0], passage_2d[:, 1], s=200, c='lightblue', 
           edgecolors='black', linewidths=2, marker='s', label='Passages', zorder=2)

# Annotate passages
for i, (x, y) in enumerate(passage_2d):
    plt.text(x, y-0.15, f'P{i}', ha='center', fontsize=10, fontweight='bold')

# Plot questions
plt.scatter(question_2d[:, 0], question_2d[:, 1], s=200, c='lightcoral', 
           edgecolors='black', linewidths=2, marker='o', label='Questions', zorder=3)

# Annotate questions
for i, (x, y) in enumerate(question_2d):
    plt.text(x, y+0.15, f'Q{i}', ha='center', fontsize=10, fontweight='bold')

# Draw connections (question to correct passage)
# 用虚线连接每个问题与其正确文档;训练好的模型应让连线两端彼此靠近
for i, (q_tokens, correct_idx) in enumerate(question_tokens):
    q_pos = question_2d[i]
    p_pos = passage_2d[correct_idx]
    plt.plot([q_pos[0], p_pos[0]], [q_pos[1], p_pos[1]], 
            'g--', alpha=0.5, linewidth=2, label='Correct' if i == 0 else '')

plt.xlabel('Dimension 1', fontsize=12)
plt.ylabel('Dimension 2', fontsize=12)
plt.title('Dense Retrieval Embedding Space (2D Projection)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nIdeal: Questions close to their relevant passages!")

## Compare with BM25 (Sparse Retrieval)(与 BM25(稀疏检索)对比)

#### 💻 代码解读

**做什么:** 实现传统的稀疏检索方法 BM25(关键词匹配打分),并和稠密检索逐题对比——这正是 DPR 论文里的核心对照实验。

**怎么做:**
- 定义 `SimpleBM25` 类:初始化时统计每个词出现在多少个文档里(`doc_freqs`)以及平均文档长度 `avg_doc_len`。
- `score` 方法按 BM25 公式给"查询-段落"打分:IDF 部分让稀有词权重更高(像"Everest"比"the"更有信息量),TF 部分用 `k1`、`b` 参数控制词频饱和与文档长度归一——本质是"数关键词重合度"。
- `retrieve` 方法给所有段落打分并返回 top-k。
- 对 5 个问题分别用 `bm25.retrieve` 和稠密检索 `retrieve_top_k` 各取 top-3,用 ✓/✗ 对比命中情况:BM25 靠字面词汇匹配(稀疏),稠密检索靠语义向量匹配(稠密)。

In [ ]:
# BM25是经典的稀疏检索基线(词面匹配),DPR论文正是要证明稠密检索能超越它
class SimpleBM25:
    """Simplified BM25 scoring"""
    def __init__(self, passages, k1=1.5, b=0.75):
        self.passages = passages
        self.k1 = k1
        self.b = b
        
        # Compute document frequencies
        self.doc_freqs = {}
        self.avg_doc_len = 0
        
        all_words = []
        # 统计文档频率df:set去重保证每篇文档对一个词最多贡献1次计数
        for passage in passages:
            words = set(passage.lower().split())
            all_words.extend(passage.lower().split())
            for word in words:
                self.doc_freqs[word] = self.doc_freqs.get(word, 0) + 1
        
        self.avg_doc_len = len(all_words) / len(passages)
        self.N = len(passages)
    
    def score(self, query, passage_idx):
        """BM25 score for query and passage"""
        query_words = query.lower().split()
        passage = self.passages[passage_idx]
        passage_words = passage.lower().split()
        passage_len = len(passage_words)
        
        # Count term frequencies
        # Counter统计词频tf:该词在这篇文档里出现的次数
        tf = Counter(passage_words)
        
        score = 0
        for word in query_words:
            if word not in tf:
                continue
            
            # IDF
            # 逆文档频率:越稀有的词权重越大;+0.5是平滑项,防止df=0或df=N时出问题
            df = self.doc_freqs.get(word, 0)
            idf = np.log((self.N - df + 0.5) / (df + 0.5) + 1)
            
            # TF component
            # b控制文档长度归一化(长文档被惩罚);k1让词频收益饱和:出现10次并不比1次好10倍
            freq = tf[word]
            norm = 1 - self.b + self.b * (passage_len / self.avg_doc_len)
            tf_component = (freq * (self.k1 + 1)) / (freq + self.k1 * norm)
            
            score += idf * tf_component
        
        return score
    
    def retrieve(self, query, k=3):
        """Retrieve top-k passages for query"""
        # 对每篇文档打分后,argsort升序再[::-1]反转取分数最高的前k篇
        scores = [self.score(query, i) for i in range(len(self.passages))]
        top_k_indices = np.argsort(scores)[::-1][:k]
        top_k_scores = [scores[i] for i in top_k_indices]
        return top_k_indices, top_k_scores

# Create BM25 retriever
bm25 = SimpleBM25(passages)

# Compare BM25 vs Dense
print("\nBM25 vs Dense Retrieval Comparison:\n" + "="*80)
for i, (question_text, correct_idx) in enumerate(questions):
    print(f"\nQ: {question_text}")
    print(f"Correct: #{correct_idx}")
    
    # BM25
    bm25_indices, bm25_scores = bm25.retrieve(question_text, k=3)
    print(f"\nBM25 Top-3:")
    for rank, (idx, score) in enumerate(zip(bm25_indices, bm25_scores), 1):
        is_correct = "✓" if idx == correct_idx else "✗"
        print(f"  {rank}. [{is_correct}] (score={score:.3f}) #{idx}")
    
    # Dense
    q_emb = question_embeddings[i]
    dense_indices, dense_scores = retrieve_top_k(q_emb, passage_embeddings, k=3)
    print(f"\nDense Top-3:")
    for rank, (idx, score) in enumerate(zip(dense_indices, dense_scores), 1):
        is_correct = "✓" if idx == correct_idx else "✗"
        print(f"  {rank}. [{is_correct}] (score={score:.3f}) #{idx}")

print("\n" + "="*80)
print("BM25: Lexical matching (sparse)")
print("Dense: Semantic matching (dense embeddings)")

## Retrieval Metrics(检索评估指标)

#### 💻 代码解读

**做什么:** 定义检索质量的两个标准评估指标——Recall@k 和 MRR,然后定量比较 BM25 和稠密检索的表现。

**怎么做:**
- 定义 `compute_metrics` 函数:对每个问题找出正确段落在检索结果中的排名;Recall@k 统计"正确答案进入前 k 名"的问题比例(k=1/3/5);MRR 取排名倒数的平均值(第 1 名得 1 分、第 2 名得 0.5 分……)——排得越靠前分越高。
- 对 5 个问题分别用 `bm25.retrieve` 和 `retrieve_top_k` 各取 top-5 预测,连同标准答案下标一起收集到 `bm25_predictions`、`dense_predictions`、`correct_indices`。
- 调用 `compute_metrics` 分别计算两种方法的指标,打印成对比表格。
- 提醒:稠密编码器未经训练,这里的数字只演示评估流程;论文中训练好的 DPR 会明显超过 BM25。

In [ ]:
def compute_metrics(predictions, correct_indices, k_values=[1, 3, 5]):
    """
    Compute retrieval metrics:
    - Recall@k: % of queries where correct passage is in top-k
    - MRR (Mean Reciprocal Rank): average 1/rank of correct passage
    """
    n_queries = len(predictions)
    
    recalls = {k: 0 for k in k_values}
    reciprocal_ranks = []
    
    # zip逐条配对(该查询的检索结果, 正确文档下标)
    for pred, correct_idx in zip(predictions, correct_indices):
        # Find rank of correct passage
        if correct_idx in pred:
            # index返回0起的位置,+1换成1起的名次;MRR取名次的倒数,排第1得1分,第2得0.5分
            rank = list(pred).index(correct_idx) + 1
            reciprocal_ranks.append(1.0 / rank)
            
            # Update recall@k
            # 正确文档排进前k名就算这次查询在Recall@k上命中
            for k in k_values:
                if rank <= k:
                    recalls[k] += 1
        else:
            reciprocal_ranks.append(0.0)
    
    # Compute averages
    # 字典推导式:把命中次数除以查询总数,得到各k值下的召回率
    mrr = np.mean(reciprocal_ranks)
    recalls = {k: v / n_queries for k, v in recalls.items()}
    
    return recalls, mrr

# Evaluate both methods
bm25_predictions = []
dense_predictions = []
correct_indices = []

for i, (question_text, correct_idx) in enumerate(questions):
    # BM25
    bm25_top, _ = bm25.retrieve(question_text, k=5)
    bm25_predictions.append(bm25_top)
    
    # Dense
    q_emb = question_embeddings[i]
    dense_top, _ = retrieve_top_k(q_emb, passage_embeddings, k=5)
    dense_predictions.append(dense_top)
    
    correct_indices.append(correct_idx)

# Compute metrics
bm25_recalls, bm25_mrr = compute_metrics(bm25_predictions, correct_indices)
dense_recalls, dense_mrr = compute_metrics(dense_predictions, correct_indices)

# Display
print("\nRetrieval Metrics:\n" + "="*60)
print(f"{'Metric':<15} {'BM25':<15} {'Dense':<15}")
print("-" * 60)
for k in [1, 3, 5]:
    print(f"Recall@{k:<10} {bm25_recalls[k]:<15.2%} {dense_recalls[k]:<15.2%}")
print(f"MRR{'':<12} {bm25_mrr:<15.3f} {dense_mrr:<15.3f}")
print("="*60)
print("\n(Models are untrained - results are random)")

## Key Takeaways(要点总结)

### Dense Passage Retrieval (DPR) Architecture:(稠密段落检索(DPR)架构)

**Dual Encoder**:
```
Question: q → BERT_Q → E_Q(q) = q_emb
Passage:  p → BERT_P → E_P(p) = p_emb

Similarity: sim(q, p) = q_emb · p_emb
```

**双编码器(Dual Encoder)**:问题和段落分别由独立的 BERT 编码器编码为稠密向量,相似度用点积计算。

### Training Objective:(训练目标)

**Contrastive Loss (InfoNCE)**:
$$
L(q_i, p_i^+, p_i^{-1}, ..., p_i^{-n}) = -\log \frac{e^{\text{sim}(q_i, p_i^+)}}{e^{\text{sim}(q_i, p_i^+)} + \sum_j e^{\text{sim}(q_i, p_i^{-j})}}
$$

Where:
- $p_i^+$: Positive (relevant) passage
- $p_i^{-j}$: Negative (irrelevant) passages

**对比损失(Contrastive Loss,InfoNCE)**,其中:
- $p_i^+$:正样本(相关)段落
- $p_i^{-j}$:负样本(不相关)段落

### In-Batch Negatives:(批内负样本)

Efficient negative mining:
```
Batch: [(q1, p1+), (q2, p2+), ..., (qB, pB+)]

For q1:
  Positive: p1+
  Negatives: p2+, p3+, ..., pB+ (from other examples)
```

**Benefits**:
- No extra passages needed
- Gradient flows through all examples
- Scales to large batch sizes

高效的负样本挖掘:对每个问题,批内其他样本的正例段落即作为其负样本。

**优点**:
- 无需额外的段落
- 梯度流经批内所有样本
- 可扩展到大批量(batch size)

### Hard Negative Mining:(困难负样本挖掘)

1. **BM25 negatives**: Top BM25 results that aren't relevant
2. **Random negatives**: Random passages from corpus
3. **In-batch negatives**: Other positives in batch

**Best**: Combine all three!

1. **BM25 负样本**:BM25 检索排名靠前但并不相关的结果
2. **随机负样本**:从语料库中随机抽取的段落
3. **批内负样本**:批内其他样本的正例

**最佳做法**:三者结合使用!

### Inference (Retrieval):(推理(检索))

**Offline**:
1. Encode all passages: $P = \{E_P(p_1), ..., E_P(p_N)\}$
2. Build MIPS index (e.g., FAISS)

**Online** (at query time):
1. Encode query: $q_{emb} = E_Q(q)$
2. Search index: top-k by $\arg\max_p \, q_{emb} \cdot p_{emb}$

**离线阶段**:
1. 编码所有段落:$P = \{E_P(p_1), ..., E_P(p_N)\}$
2. 构建 MIPS 索引(例如 FAISS)

**在线阶段**(查询时):
1. 编码查询:$q_{emb} = E_Q(q)$
2. 检索索引:按 $\arg\max_p \, q_{emb} \cdot p_{emb}$ 取 top-k

### DPR vs BM25:(DPR 与 BM25 对比)

| Aspect | BM25 | DPR |
|--------|------|-----|
| Matching | Lexical (exact words) | Semantic (meaning) |
| Training | None (heuristic) | Learned from data |
| Robustness | Sensitive to wording | Handles paraphrases |
| Speed | Fast (sparse) | Fast with MIPS index |
| Memory | Low | High (dense vectors) |

| 方面 | BM25 | DPR |
|--------|------|-----|
| 匹配方式 | 词面匹配(精确词) | 语义匹配(含义) |
| 训练 | 无需训练(启发式) | 从数据中学习 |
| 鲁棒性 | 对措辞敏感 | 能处理同义改写 |
| 速度 | 快(稀疏) | 借助 MIPS 索引同样很快 |
| 内存 | 低 | 高(稠密向量) |

### Results (from paper):(论文中的结果)

**Natural Questions**:
- BM25: 59.1% Top-20 accuracy
- DPR: 78.4% Top-20 accuracy

**WebQuestions**:
- BM25: 55.0%
- DPR: 75.0%

**TREC**:
- BM25: 70.9%
- DPR: 79.4%

在 Natural Questions 上,DPR 的 Top-20 准确率为 78.4%,显著超过 BM25 的 59.1%;在 WebQuestions(75.0% vs 55.0%)和 TREC(79.4% vs 70.9%)上同样大幅领先。

### Implementation Details:(实现细节)

1. **Encoders**: BERT-base (110M params)
2. **Embedding dim**: 768 (BERT hidden size)
3. **Batch size**: 128 (large for in-batch negatives)
4. **Hard negatives**: 1 BM25 + 1 random per positive
5. **Training**: ~40 epochs on 59k QA pairs

1. **编码器**:BERT-base(1.1 亿参数)
2. **嵌入维度**:768(BERT 隐藏层大小)
3. **批量大小**:128(大批量可提供更多批内负样本)
4. **困难负样本**:每个正例配 1 个 BM25 负样本 + 1 个随机负样本
5. **训练**:在 5.9 万个 QA 对上训练约 40 个 epoch

### Advantages:(优势)

- ✅ **Semantic matching**: Understands meaning, not just words
- ✅ **End-to-end**: Learned from question-passage pairs
- ✅ **Handles paraphrases**: "tallest mountain" = "highest peak"
- ✅ **Scalable**: MIPS with FAISS for billions of passages
- ✅ **Outperforms BM25**: +15-20% absolute accuracy

- ✅ **语义匹配**:理解含义,而不仅是词面
- ✅ **端到端**:直接从问题-段落对中学习
- ✅ **处理同义改写**:"tallest mountain" = "highest peak"
- ✅ **可扩展**:借助 FAISS 做 MIPS,可支持数十亿段落
- ✅ **优于 BM25**:绝对准确率提升 15-20%

### Limitations:(局限性)

- ❌ **Requires training data**: Need QA pairs
- ❌ **Memory**: Dense vectors for all passages
- ❌ **Index updates**: Re-encode when corpus changes
- ❌ **May miss exact matches**: BM25 better for rare entities

- ❌ **需要训练数据**:需要 QA 对
- ❌ **内存开销**:需为所有段落存储稠密向量
- ❌ **索引更新**:语料库变化时需要重新编码
- ❌ **可能漏掉精确匹配**:对稀有实体,BM25 表现更好

### Best Practices:(最佳实践)

1. **Hybrid retrieval**: Combine BM25 + DPR
2. **Large batches**: More in-batch negatives
3. **Hard negatives**: Use BM25 top results
4. **Fine-tune**: Domain-specific data improves results
5. **FAISS**: Use for fast MIPS at scale

1. **混合检索**:结合 BM25 与 DPR
2. **大批量**:获得更多批内负样本
3. **困难负样本**:使用 BM25 排名靠前的结果
4. **微调**:领域特定数据可提升效果
5. **FAISS**:用于大规模下的快速 MIPS

### Modern Extensions:(现代扩展)

- **ColBERT**: Late interaction for better ranking
- **ANCE**: Approximate nearest neighbor negatives
- **RocketQA**: Cross-batch negatives
- **Contriever**: Unsupervised dense retrieval
- **Dense X Retrieval**: Multi-vector representations

- **ColBERT**:后期交互(late interaction),排序效果更好
- **ANCE**:基于近似最近邻的负样本
- **RocketQA**:跨批次负样本
- **Contriever**:无监督稠密检索
- **Dense X Retrieval**:多向量表示

### Applications:(应用)

- Open-domain QA (e.g., Google search)
- RAG (Retrieval-Augmented Generation)
- Document search
- Semantic search
- Knowledge base completion

- 开放域问答(如 Google 搜索)
- RAG(检索增强生成)
- 文档搜索
- 语义搜索
- 知识库补全